In [ ]:
!pip install numpy pandas matplotlib scipy yfinance arch statsmodels -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Chart style
BLUE = '#1A3A6E'; RED = '#DC3545'; GREEN = '#2E7D32'
ORANGE = '#E67E22'; GRAY = '#666666'; PURPLE = '#8E44AD'

plt.rcParams.update({
    'figure.facecolor': 'none', 'axes.facecolor': 'none',
    'savefig.facecolor': 'none', 'savefig.transparent': True,
    'axes.grid': False, 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'xtick.labelsize': 9,
    'ytick.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 1.0, 'axes.linewidth': 0.6,
    'legend.facecolor': 'none', 'legend.framealpha': 0, 'legend.edgecolor': 'none',
    'figure.dpi': 150,
})

def save_chart(fig, name):
    fig.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True, dpi=150)
    fig.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved: {name}')

def legend_bottom(ax, ncol=2, y=-0.18):
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

In [ ]:
# ============================================================
# Normal, Student-t, and GED distribution comparison
# ============================================================
from scipy.special import gamma as gammafn

x = np.linspace(-6, 6, 1000)

# --- Normal PDF ---
normal_pdf = stats.norm.pdf(x, 0, 1)

# --- Student-t PDFs for various df ---
t_dfs = [3, 5, 10, 30]

# --- GED PDF ---
def ged_pdf(x, nu):
    lam = np.sqrt(2**(-2/nu) * gammafn(1/nu) / gammafn(3/nu))
    return (nu * np.exp(-0.5 * np.abs(x/lam)**nu)) / (lam * 2**(1+1/nu) * gammafn(1/nu))

ged_nus = [1.0, 1.5, 2.0, 3.0]

# ===== Plot 1: Normal distribution =====
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, normal_pdf, color=BLUE, lw=2, label='N(0,1)')
ax.fill_between(x, normal_pdf, alpha=0.15, color=BLUE)
# Mark tails
ax.fill_between(x[x < -2], normal_pdf[x < -2], alpha=0.3, color=RED, label='Tails (|x|>2)')
ax.fill_between(x[x > 2], normal_pdf[x > 2], alpha=0.3, color=RED)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Normal (Gaussian) Distribution', fontweight='bold')
ax.legend(frameon=False)
save_chart(fig, 'garch_dist_normal')
plt.show()

In [ ]:
# ===== Plot 2: Student-t distribution =====
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, normal_pdf, color=GRAY, lw=1.5, ls='--', label='Normal')
colors_t = [RED, ORANGE, GREEN, BLUE]
for df, c in zip(t_dfs, colors_t):
    ax.plot(x, stats.t.pdf(x, df), color=c, lw=1.5, label=f'Student-t (ν={df})')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Student-t Distribution — Effect of Degrees of Freedom', fontweight='bold')
legend_bottom(ax, ncol=5, y=-0.15)
save_chart(fig, 'garch_dist_student')
plt.show()

In [ ]:
# ===== Plot 3: GED distribution =====
fig, ax = plt.subplots(figsize=(10, 5))
colors_g = [RED, ORANGE, BLUE, GREEN]
labels_g = ['Laplace (ν=1)', 'GED (ν=1.5)', 'Normal (ν=2)', 'GED (ν=3)']
for nu, c, lab in zip(ged_nus, colors_g, labels_g):
    ax.plot(x, ged_pdf(x, nu), color=c, lw=1.5, label=lab)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Generalized Error Distribution (GED)', fontweight='bold')
legend_bottom(ax, ncol=4, y=-0.15)
save_chart(fig, 'garch_dist_ged')
plt.show()

print("\nKurtosis comparison:")
print(f"  Normal:      κ = 3.00")
for df in t_dfs:
    if df > 4:
        print(f"  Student-t(ν={df}): κ = {3*(df-2)/(df-4):.2f}")
    else:
        print(f"  Student-t(ν={df}): κ = ∞ (ν≤4)")
for nu in ged_nus:
    k = gammafn(1/nu)*gammafn(5/nu) / gammafn(3/nu)**2
    print(f"  GED(ν={nu}):    κ = {k:.2f}")